In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
n = 500

klienci = ["Jan Kowalski", "Anna Nowak", " PIOTR WIŚNIEWSKI", "marta wójcik", 
           " jan kowalski", "Tomasz Kaczmarek", "anna nowak ", "Zofia Król", 
           "kowalska ", "Krzysztof Kamiński", " Magdalena Dąbrowska"]
produkty = ["Laptop", "Mysz", "Klawiatura", "Monitor", "laptop", "MYSZ", 
            "Słuchawki", "Pendrive", "monitor", "Webcam"]
kategorie = ["Elektronika", "elektronika", "ELEKTRONIKA", "Akcesoria", 
             "akcesoria", "Akcesoria "]
miasta = ["Warszawa", "Kraków", "warszawa", "Gdańsk", "WROCŁAW", 
          "Poznań", "Łódź ", " Warszawa", "kraków"]

start_date = datetime(2025, 1, 1)
daty_iso = [(start_date + timedelta(days=int(d))).strftime("%Y-%m-%d") for d in np.random.randint(0, 300, n // 2)]
daty_pl = [(start_date + timedelta(days=int(d))).strftime("%d.%m.%Y") for d in np.random.randint(0, 300, n // 2)]
daty = daty_iso + daty_pl
np.random.shuffle(daty)

df = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "klient": np.random.choice(klienci, n),
    "produkt": np.random.choice(produkty, n),
    "kategoria": np.random.choice(kategorie, n),
    "miasto": np.random.choice(miasta, n),
    "ilosc": np.random.choice([1, 2, 3, 5, -1, 0], n, p=[0.5, 0.2, 0.15, 0.1, 0.025, 0.025]),
    "cena_jednostkowa": np.random.choice(
        ["199.99", "299,99", "1 499.00", "89.50", "2999", "399.00 zł", None, "abc"], n
    ),
    "data_zamowienia": daty,
    "email": np.random.choice(
        ["anna@gmail.com", "JAN@WP.PL", "piotr.w@onet", "marta@gmail.com", 
         "tomasz@interia.pl", None, "krzysztof.k@gmail.com", "brak"], n
    )
})

for col in ["miasto", "kategoria", "data_zamowienia"]:
    df.loc[df.sample(frac=0.05, random_state=1).index, col] = np.nan

df = pd.concat([df, df.sample(20, random_state=2)], ignore_index=True)
df.to_csv("zamowienia_messy.csv", index=False)
print(f"Wygenerowano plik 'zamowienia_messy.csv' — {len(df)} wierszy")

In [ ]:
df = pd.read_csv("zamowienia_messy.csv")


print("Wymiary (shape):", df.shape)
print("\n--- INFO ---")
df.info()
print("\n--- BRAKUJĄCE WARTOŚCI (NaN) ---")
print(df.isnull().sum())
print("\n--- PRZYKŁADOWE KATEGORIE ---")
print(df["kategoria"].value_counts())


In [ ]:
import numpy as np

df = df.drop_duplicates()

df["klient"] = df["klient"].str.strip().str.title()
df["produkt"] = df["produkt"].str.strip().str.capitalize()
df["kategoria"] = df["kategoria"].str.strip().str.lower()
df["miasto"] = df["miasto"].str.strip().str.title()

df["data_zamowienia"] = pd.to_datetime(df["data_zamowienia"], format='mixed', dayfirst=True, errors='coerce')

df["cena_jednostkowa"] = df["cena_jednostkowa"].astype(str)
df["cena_jednostkowa"] = df["cena_jednostkowa"].str.replace(r"[^\d,\.]", "", regex=True)
df["cena_jednostkowa"] = df["cena_jednostkowa"].str.replace(",", ".")
df["cena_jednostkowa"] = pd.to_numeric(df["cena_jednostkowa"], errors="coerce")

df = df.dropna(subset=["cena_jednostkowa", "data_zamowienia"])
df["miasto"] = df["miasto"].fillna("unknown")
df["kategoria"] = df["kategoria"].fillna("unknown")
df["email"] = df["email"].fillna("brak_emaila")

df = df[df["ilosc"] > 0]

print("Po czyszczeniu rozmiar danych to:", df.shape)
display(df.head())

In [ ]:
df["wartosc_zamowienia"] = df["ilosc"] * df["cena_jednostkowa"]

df["rok"] = df["data_zamowienia"].dt.year
df["miesiac"] = df["data_zamowienia"].dt.month
df["nazwa_dnia"] = df["data_zamowienia"].dt.day_name()

df["email_poprawny"] = df["email"].str.match(r".+@.+\..+", na=False)

display(df[["klient", "email", "email_poprawny", "data_zamowienia", "nazwa_dnia", "wartosc_zamowienia"]].head())

In [ ]:
import matplotlib.pyplot as plt

wartosc_miesiace = df.groupby("miesiac")["wartosc_zamowienia"].sum().reset_index()
print("1. Łączna wartość zamówień w każdym miesiącu:")
display(wartosc_miesiace)

top_klienci = df.groupby("klient")["wartosc_zamowienia"].sum().sort_values(ascending=False).head(5).reset_index()
print("\n2. Top 5 klientów (największa łączna wartość):")
display(top_klienci)

srednia_kategoria = df.groupby("kategoria")["wartosc_zamowienia"].mean().reset_index()
print("\n3. Średnia wartość zamówienia w każdej kategorii:")
display(srednia_kategoria)


plt.figure(figsize=(10, 6))
plt.bar(wartosc_miesiace["miesiac"].astype(str), wartosc_miesiace["wartosc_zamowienia"], color='#1f77b4')
plt.title("Łączna wartość zamówień w miesiącach (2025)", fontsize=14)
plt.xlabel("Miesiąc", fontsize=12)
plt.ylabel("Łączna wartość (PLN)", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

df.to_csv("zamowienia_clean.csv", index=False)
print("Sukces! Oczyszczone dane zostały zapisane jako 'zamowienia_clean.csv'.")